# EMS741 Few-Shot Segmentation: Reptile

This notebook is formatted to run correctly on both JupyterHub and Colab by relying on environment variables for the dataset and importing all core functions from `core_methods.py`.

In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import os, zipfile, subprocess

# If the dataset is already available on the cluster, set EMS741_DATA_ROOT
# before starting Jupyter. Otherwise, it will download into the CWD.
DATA_ROOT = None

if DATA_ROOT is None:
    env_root = os.environ.get("EMS741_DATA_ROOT")
    DATA_ROOT = Path(env_root) if env_root else Path.cwd()

def has_dataset(root: Path):
    return all((root / s).exists() for s in ["train", "val", "test"])

if not has_dataset(DATA_ROOT):
    print(f"Dataset not found in {DATA_ROOT.resolve()}")
    print("Downloading dataset to current working directory...")
    url = "https://zenodo.org/records/18745413/files/ems741_cw_data.zip?download=1"
    zip_path = Path("data.zip")
    if not zip_path.exists():
        subprocess.run(["wget", "-O", str(zip_path), url], check=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(Path.cwd())
    DATA_ROOT = Path.cwd()

print("DATA_ROOT:", DATA_ROOT.resolve())

DATA_ROOT: /home/jovyan/Git/QMUL-EMS741-Group-8


In [2]:
import torch
import numpy as np
import random

from core_methods import (
    UNet,
    SegDataset,
    FewShotEpisode,
    discover_tasks,
    reptile_meta_train,
    adapt_and_evaluate,
    train_baseline,
    unified_adapt_and_evaluate,
    bce_dice_loss,
    dice_score,
)

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cuda


In [3]:
train_tasks = discover_tasks(DATA_ROOT / "train")
val_tasks   = discover_tasks(DATA_ROOT / "val")
test_tasks  = discover_tasks(DATA_ROOT / "test")

print("Train tasks:", list(train_tasks.keys()))
print("Val tasks:",   list(val_tasks.keys()))
print("Test tasks:",  list(test_tasks.keys()))

Train tasks: ['task_2', 'task_3', 'task_5', 'task_7']
Val tasks: ['task_4', 'task_6']
Test tasks: ['task_1', 'task_8']


In [4]:
N_OUTER   = 1
K_INNER   = 5
INNER_LR  = 1e-3
META_LR   = 0.1
VAL_EVERY = 200

print("Starting Reptile meta-training...")
meta_model, meta_history = reptile_meta_train(
    train_tasks=train_tasks,
    val_tasks=val_tasks,
    n_outer=N_OUTER,
    k_inner=K_INNER,
    inner_lr=INNER_LR,
    meta_lr=META_LR,
    batch_size=4,
    val_every=VAL_EVERY,
    n_shot_val=5,
)

torch.save(meta_model.state_dict(), "meta_model.pth")
print("Meta-model saved to meta_model.pth")

Starting Reptile meta-training...
Meta-model saved to meta_model.pth


In [5]:
N_SHOTS         = [1, 3, 5]
ADAPT_STEPS     = 20
BASELINE_EPOCHS = 20   # relatively small, as discussed

for n_shot in N_SHOTS:
    print(f"\n=== {n_shot}-shot evaluation ===")
    for task_name, task_dict in test_tasks.items():
        # Reptile adaptation
        r_dice, adapted_model, _ = adapt_and_evaluate(
            meta_model,
            task_dict,
            n_shot=n_shot,
            adapt_steps=ADAPT_STEPS,
            adapt_lr=1e-3,
            seed=42,
        )

        # Baseline: random UNet adapted with unified_adapt_and_evaluate
        base = UNet().to(DEVICE)
        b_dice, baseline_model, _ = unified_adapt_and_evaluate(
            base,
            task_dict,
            n_shot=n_shot,
            epochs=BASELINE_EPOCHS,
            lr=1e-3,
            seed=42,
        )

        print(f"Task {task_name} | Reptile: {r_dice:.4f} | Baseline: {b_dice:.4f}")


=== 1-shot evaluation ===
Task task_1 | Reptile: 0.1089 | Baseline: 0.0555
Task task_8 | Reptile: 0.0052 | Baseline: 0.0126

=== 3-shot evaluation ===
Task task_1 | Reptile: 0.0468 | Baseline: 0.0066
Task task_8 | Reptile: 0.0162 | Baseline: 0.0060

=== 5-shot evaluation ===
Task task_1 | Reptile: 0.2779 | Baseline: 0.2852
Task task_8 | Reptile: 0.0352 | Baseline: 0.0121
